# Cross-sectional selection on a **shipped panel** — the worker side of `kgpu` panel mode

`RUN__feature_importance_report.ipynb` reads pools through `UnifiedSchemaReader`, which is
why a Kaggle worker can run it at all: `kgpu` swaps that class for a parquet-backed
subclass and every pool arrives as a file.

⚠️ **A CROSS-SECTIONAL TARGET CANNOT TAKE THAT ROUTE, AND NOT FOR WANT OF A PARAMETER.**
`feature_selection.cross_sectional.read_universe_panel` builds `pool__basic ⋈ pool__targets`
with one hand-written SQL statement and derives `cs_rank_{h}day` from it, so it reaches for
`reader.driver._cursor_ctx()` — and `ParquetSchemaReader.driver` answers *"there is no
database on a Kaggle worker"*. The cross-sectional read **bypasses every abstraction the
parquet payload replaces** (`CSP-1` in its second form).

So the join runs where the database is — `kgpu export._export_panel`, on the machine that
can see `unified_schema_all` — and this notebook receives one finished `panel.parquet` with
`cs_rank` already derived, plus the provenance the reader would have supplied: the schema,
the channel→pool map, and the universe.

**Nothing below re-implements a selection.** It hands the panel to
`feature_selection.run.run_selection`, which is the same function
`python -m feature_selection.run --target cs_rank_20day` calls — same selector, same purged
by-date CV, same panel-aware null, same `write_report`, same `outstanding.csv`. That is what
makes a Kaggle panel run comparable with a local one at all.

⚠️ **Two things that are NOT the same, both recorded in `metadata.json` and both in
`contract.SETUP_KEYS`, so they can never be unioned into one `__final__` table by accident:**
`env_fingerprint` (Kaggle ships xgboost 3.2.0 / sklearn 1.6.1 against `mt_env`'s 2.1.1 /
1.7.2 — a major version of a ranker) and `design_dtype`.

⚠️ **The shipped `cs_rank` is a rank within the SHIPPED names, not within all 781.** That is
the intended experiment — a tradeable liquid cross-section — and the manifest says so.


In [ ]:
import os
import re
import sys

# ⚠️ EITHER WORKING DIRECTORY MUST WORK — the repo root or this notebook's own folder,
# and on a Kaggle worker neither is a repo checkout: `kgpu_bootstrap` unpacks the shipped
# source into /kaggle/working/src, so walking UP from the CWD finds /kaggle/working. The
# same walk finds the real repo locally. `report.write_report` anchors a relative
# REPORT_ROOT to its own three-levels-up, which is why the source must land there and
# not in a library folder.
def _find_repo_root() -> str:
    here = os.path.abspath(os.getcwd())
    while True:
        if os.path.isdir(os.path.join(here, "src", "feature_selection")):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            raise RuntimeError(
                f"no repo root above {os.getcwd()!r} - expected a folder holding "
                f"src/feature_selection. On a worker this means kgpu_bootstrap.setup() "
                f"did not run: check cell 0 and the mounted payload."
            )
        here = parent


REPO_ROOT = _find_repo_root()
if os.path.join(REPO_ROOT, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
print("repo root:", REPO_ROOT)

import pandas as pd
from IPython.display import Markdown, display

from feature_selection import contract, cross_sectional, plots, report, run
from feature_selection import selector as selector_methods

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
plots.use_theme()

In [ ]:
TICKER = "ALL"  # the schema the panel was exported FROM — unified_schema_<ticker>

# ── THE THREE KNOBS YOU CHANGE ───────────────────────────────────────────────
TARGET = "cs_rank_20day"  # ⚠️ must carry the `cs_` prefix — that prefix IS the switch
HORIZON = 20  # h — ⚠️ MUST match the target's own horizon (the `20` in its name)
LOOKBACK = 20  # d — window length in SESSIONS
# ─────────────────────────────────────────────────────────────────────────────
#
# ⚠️ **h=20 IS THE POINT OF THIS RUN, NOT AN INCIDENTAL SETTING.** CLAUDE.md §2a-bis:
# the one block that ever beat its benchmark (`controls` — momentum + liquidity, no
# text) only does so from 4 weeks out, and LOSES at 5-10 sessions on the most liquid
# names. Four independent threads have failed at h=5. This asks whether the HORIZON
# was the variable, at the one grain (§2b: ~100+ names) that has ever cleared a null.
#
# ⚠️ At h=20 the independent count is `n_dates / h` = ~213 at EITHER grain. What
# differs is the QUALITY of each observation: one stock's ±1 sign has sd ~1.0, an IC
# over ~300 names has sd ~0.06. Width buys precision per observation (1/√N), never
# more observations — which is why h=20 kills a single-stock study and does not kill
# this one.

# The payload directory holding `manifest.json` + `panel.parquet`.
# "" = discover the dataset Kaggle mounted. Set it to a local `.payload/<job>/` folder
# to run this notebook off Kaggle against the same artefact.
PANEL_PAYLOAD = ""

NORMALIZE = "none"  # "none" | "zscore" | "window_relative" — WITHIN each window
FEATURE_NORMALIZE = "cs_rank"  # ⚠️ cross-sectional only: rank each feature within its
# date BEFORE windowing. It removes the level that acts as a date proxy and the size
# that acts as a permanent stock label — see cross_sectional.py §3.
MIN_IC_WIDTH = 5  # dates narrower than this contribute no IC and no rank
MAX_FEATURES = None  # None = the per-run MEASURED cut (selection_cut, STL-1)
CORR_THRESHOLD = 0.9
N_SPLITS = 5
MIN_TRAIN = 500  # ⚠️ SESSIONS here, not rows — PurgedWalkForwardByDate
DEVICE = "cuda"  # ⚠️ part of the SETUP, not a speed knob — gpu.py §1
DESIGN_DTYPE = "float64"  # ⚠️ never float32 for a number you will quote: TODO P0-3
# measured a 52% relative change in `ic_mean` from the dtype alone.
RANDOM_STATE = 18  # the SELECTOR seed; the null draws from run.NULL_SEED = 7
METHODS = None  # None = feature_selection.selector.METHODS, the measured three

RUN_STABILITY = True  # per-fold SHAP ranking (cheap)
RUN_NULL = False  # ⚠️ the BAR. Each draw re-runs the whole selection.
N_NULL = 20
HOLDOUT_START = None  # e.g. "2024-06-01"; None = no holdout

TOP_N_FIGURES = 30
REPORT_ROOT = "reports/feature_selection"  # ⚠️ relative, anchored to the REPO
NOTES = ""

# ⚠️ The horizon is checked against the TARGET'S OWN NAME rather than trusted. `d` and
# `h` flow from a table name into a dataset and into a model config all the way down
# this chain; a run whose HORIZON disagrees with its label is mislabelled everywhere
# downstream and nothing later can detect it.
_named = re.search(r"(\d+)day$", TARGET)
if not TARGET.startswith("cs_"):
    raise ValueError(
        f"TARGET {TARGET!r} has no `cs_` prefix, so run_selection would take the "
        f"SINGLE-SERIES path over a 300-name panel — pooling cross-sectional with "
        f"time-series variation and counting 300 stocks on one date as 300 "
        f"observations (issue PNL-1's shape, at the selection stage)."
    )
if not _named or int(_named.group(1)) != HORIZON:
    raise ValueError(
        f"TARGET {TARGET!r} and HORIZON {HORIZON} disagree. The label is built at the "
        f"horizon in its own name; the purge gap, `n_eff` and every error bar below "
        f"are computed from HORIZON."
    )
print(f"{TICKER} · {TARGET} · d={LOOKBACK} h={HORIZON} · device={DEVICE} · "
      f"null={'%d draws' % N_NULL if RUN_NULL else 'NONE (evidence=no_null)'}")

In [ ]:
# ⚠️ **THE PAYLOAD IS FOUND BY CONTENT, AT WHATEVER DEPTH KAGGLE CHOSE.** The mount is
# `/kaggle/input/datasets/<owner>/<slug>/`, not `/kaggle/input/<slug>/` — measured
# 2026-08-15, and it cost a run. `kgpu_bootstrap.find_payload` is the same search the
# injected first cell used, so this cannot disagree with what was already loaded.
#
# Off Kaggle the two remote-side modules are not on the path (they travel FLAT in the
# payload, not inside `source.zip`), so the repo copy is appended — appended, never
# inserted, so a mounted payload's own copy still wins on a worker.
_remote = os.path.join(REPO_ROOT, "src", "kaggle_gpu", "kgpu", "remote")
if os.path.isdir(_remote) and _remote not in sys.path:
    sys.path.append(_remote)

import kgpu_bootstrap
import kgpu_remote_reader

payload = PANEL_PAYLOAD or kgpu_bootstrap.find_payload()
if not payload:
    raise RuntimeError(
        "no kgpu payload is mounted and PANEL_PAYLOAD is empty.\n"
        "  On a worker this means the kernel's dataset_sources are wrong.\n"
        "  Locally, point PANEL_PAYLOAD at src/kaggle_gpu/.payload/<job>/ after\n"
        "  running: python -m kgpu export <job>"
    )

provided = kgpu_remote_reader.load_panel(payload)
panel = provided.frame
print(f"payload  : {payload}")
print(f"provenance: {provided.note}")
print(f"panel    : {len(panel):,} x {panel.shape[1]}  "
      f"({panel['date'].min():%Y-%m-%d} -> {panel['date'].max():%Y-%m-%d})")
print(f"universe : {len(provided.universe or [])} names, "
      f"cs_rank is a rank WITHIN them")
print(f"channels : " + ", ".join(
    f"{table} {len(cols)}" for table, cols in provided.columns_by_table.items()))
print(f"memory   : {panel.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

In [ ]:
# ⚠️ **THE SHAPE NUMBERS EVERY CLAIM BELOW RESTS ON, READ BEFORE THE FIT.** `n_eff` is
# `n_dates / h` on a panel — NOT `n_rows / h`. 300 stocks on one Tuesday are ONE
# observation of the market; the cross-section buys precision per observation and
# nothing else (CLAUDE.md §5 rule 7).
summary = cross_sectional.panel_summary(panel, TARGET)
print(summary.to_string())
print(f"\nn_eff = dates / h = {summary['dates']} / {HORIZON} = "
      f"{summary['dates'] / HORIZON:.0f} independent observations")
print(f"purge gap = d + h - 1 = {LOOKBACK} + {HORIZON} - 1 = "
      f"{LOOKBACK + HORIZON - 1} SESSIONS at each fold boundary")
summary.to_frame().T

In [ ]:
# ⚠️ **THE SAME FUNCTION `python -m feature_selection.run` CALLS.** `provided_panel`
# replaces the READ and nothing else: the selector, the purged by-date CV, the
# panel-aware `date_block` null, `write_report` and the `outstanding.csv` handoff are
# the code that runs locally. A worker-only copy of any of it would be a second place
# for the procedure to drift, and two studies that drifted cannot be set side by side.
#
# ⚠️ It writes the report BEFORE printing anything about it — the null is the expensive
# artefact and a KeyError in a summary f-string has thrown twenty completed draws away
# before (CLAUDE.md §5 rule 20).
written = run.run_selection(
    ticker=TICKER,
    target=TARGET,
    lookback=LOOKBACK,
    horizon=HORIZON,
    normalize=NORMALIZE,
    design_dtype=DESIGN_DTYPE,
    max_features=MAX_FEATURES,
    corr_threshold=CORR_THRESHOLD,
    n_splits=N_SPLITS,
    min_train=MIN_TRAIN,
    device=DEVICE,
    random_state=RANDOM_STATE,
    stability=RUN_STABILITY,
    null_draws=N_NULL if RUN_NULL else 0,
    holdout_start=HOLDOUT_START,
    root=REPORT_ROOT,
    notes=NOTES,
    top=TOP_N_FIGURES,
    feature_normalize=FEATURE_NORMALIZE,
    min_ic_width=MIN_IC_WIDTH,
    methods=METHODS if METHODS else selector_methods.METHODS,
    provided_panel=provided,
)

In [ ]:
# ⚠️ **THE RUN FOLDER IS THE DELIVERABLE, AND `outstanding.csv` IS WHAT MAKES IT
# VISIBLE.** `final_features.plan_from_reports` skips a folder carrying no shortlist
# WITHOUT A WORD — measured 2026-08-15, when the two newest runs, both produced through
# `kgpu`, sat in exactly that state while `final_features` planned 19 runs and reported
# no error. `run_selection` writes it above; this checks what came home.
RUN_ROOT = os.path.dirname(written.path)
shortlist = os.path.join(written.path, contract.SHORTLIST_FILENAME)
if not os.path.exists(shortlist):
    raise RuntimeError(
        f"no {contract.SHORTLIST_FILENAME} in {written.path} — the WARNING lines above "
        f"say why. `final_features` cannot see this run until it exists; the report "
        f"folder itself is already on disk and is not lost."
    )
print(contract.describe(RUN_ROOT))
pd.read_csv(shortlist)

In [ ]:
display(Markdown(open(os.path.join(written.path, "README.md"), encoding="utf-8").read()))